# BraTS2020 - Integration Smoke Test

Verify that the modular GitHub repository works in a fresh Kaggle environment.

Tests:
- Repository cloning
- Python module imports
- Dice metrics
- 3D U-Net forward pass
- Full-volume validation
- Training and checkpoint functions

No full training or test evaluation is performed.

In [ ]:
# STEP 8.5A - ENVIRONMENT SETUP AND MODULE IMPORTS

import sys
import subprocess
import importlib.util
from pathlib import Path

# ==========================================
# 1. INSTALL MONAI IF NEEDED
# ==========================================

if importlib.util.find_spec("monai") is None:
    print("Installing MONAI...")

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "monai"],
        check=True,
    )

import torch
import monai

# ==========================================
# 2. CLONE THE GITHUB REPOSITORY
# ==========================================

REPO_URL = (
    "https://github.com/"
    "tthanh81/brats2020-segmentation.git"
)

BRANCH = "refactor/data-pipeline"

PROJECT_ROOT = Path(
    "/kaggle/working/brats2020-integration-check"
)

if not PROJECT_ROOT.exists():

    subprocess.run(
        [
            "git", "clone",
            "--branch", BRANCH,
            "--single-branch",
            REPO_URL,
            str(PROJECT_ROOT),
        ],
        check=True,
    )

else:
    subprocess.run(
        ["git", "pull", "--ff-only"],
        cwd=PROJECT_ROOT,
        check=True,
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ==========================================
# 3. IMPORT PROJECT MODULES
# ==========================================

from src.data.dataset import BraTSVolumeDataset
from src.data.patches import RandomPatchDataset

from src.models.unet3d import (
    build_unet3d,
    build_loss,
)

from src.training.engine import train_one_epoch

from src.training.checkpoints import (
    save_checkpoint,
    load_checkpoint,
)

from src.evaluation.metrics import (
    region_masks,
    binary_dice,
    brats_dice_scores,
)

from src.evaluation.validation import validate_model

# ==========================================
# 4. VERIFY REPOSITORY VERSION
# ==========================================

commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"],
    cwd=PROJECT_ROOT,
    text=True,
).strip()

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("\n" + "=" * 55)
print("INTEGRATION ENVIRONMENT SUMMARY")
print("=" * 55)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("MONAI:", monai.__version__)

print("\nGitHub:")
print("Branch:", BRANCH)
print("Commit:", commit)

print("\nHardware:")
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nRepository modules:")
print("Data pipeline: PASS")
print("Model: PASS")
print("Training engine: PASS")
print("Checkpoint utilities: PASS")
print("Dice metrics: PASS")
print("Validation: PASS")

print("\nPASS: Modular repository environment ready.")

Installing MONAI...


Cloning into '/kaggle/working/brats2020-integration-check'...



INTEGRATION ENVIRONMENT SUMMARY
Python: 3.13.15
PyTorch: 2.11.0+cu128
MONAI: 1.6.1

GitHub:
Branch: refactor/data-pipeline
Commit: 5d7ce7d

Hardware:
Device: cuda
GPU: Tesla T4

Repository modules:
Data pipeline: PASS
Model: PASS
Training engine: PASS
Checkpoint utilities: PASS
Dice metrics: PASS
Validation: PASS

PASS: Modular repository environment ready.


In [ ]:
# STEP 8.5B - DICE METRICS + MODEL FORWARD + LOSS

import torch

print("=" * 60)
print("STEP 8.5B - MODEL AND METRICS SMOKE TEST")
print("=" * 60)

# ==========================================
# TEST 1 - DICE METRICS
# ==========================================

ground_truth = torch.tensor([
    [[0, 1], [2, 3]],
    [[0, 0], [1, 2]],
], dtype=torch.uint8)

# Case 1: Perfect prediction
scores_perfect = brats_dice_scores(
    ground_truth,
    ground_truth.clone(),
)

assert all(
    abs(score - 1.0) < 1e-8
    for score in scores_perfect.values()
)

print("\nTEST 1A - Perfect prediction")
print("Dice:", scores_perfect)
print("PASS: Perfect prediction gives Dice = 1.")

# Case 2: Both masks have no tumor
empty = torch.zeros((2, 2, 2), dtype=torch.uint8)

scores_empty = brats_dice_scores(empty, empty)

assert all(
    score == 1.0
    for score in scores_empty.values()
)

print("\nTEST 1B - Both masks empty")
print("Dice:", scores_empty)
print("PASS: Empty-mask convention verified.")

# Case 3: False-positive enhancing tumor
false_positive = empty.clone()
false_positive[0, 0, 0] = 3

scores_fp = brats_dice_scores(
    false_positive,
    empty,
)

assert scores_fp["ET"] == 0.0

print("\nTEST 1C - False-positive ET")
print("Dice:", scores_fp)
print("PASS: False-positive ET gives ET Dice = 0.")


# ==========================================
# TEST 2 - BUILD 3D U-NET
# ==========================================

model = build_unet3d().to(device)
model.eval()

num_parameters = sum(
    p.numel()
    for p in model.parameters()
)

assert num_parameters == 1_191_413

print("\nTEST 2 - Model architecture")
print("Parameters:", f"{num_parameters:,}")
print("PASS: Model architecture verified.")


# ==========================================
# TEST 3 - FORWARD PASS
# ==========================================

torch.manual_seed(42)

images = torch.randn(
    1, 4, 32, 32, 32,
    dtype=torch.float32,
    device=device,
)

targets = torch.randint(
    low=0,
    high=4,
    size=(1, 1, 32, 32, 32),
    dtype=torch.long,
    device=device,
)

with torch.inference_mode():
    logits = model(images)

assert logits.shape == (1, 4, 32, 32, 32)
assert torch.isfinite(logits).all().item()

print("\nTEST 3 - Forward pass")
print("Input shape:", tuple(images.shape))
print("Output shape:", tuple(logits.shape))
print("PASS: Forward pass verified.")


# ==========================================
# TEST 4 - DICE + CROSS ENTROPY LOSS
# ==========================================

criterion = build_loss().to(device)

with torch.inference_mode():
    loss = criterion(logits.float(), targets)

assert torch.isfinite(loss).item()
assert loss.item() >= 0

print("\nTEST 4 - DiceCELoss")
print("Loss:", round(loss.item(), 6))
print("PASS: Loss computation verified.")


# ==========================================
# FINAL SUMMARY
# ==========================================

print("\n" + "=" * 60)
print("STEP 8.5B SUMMARY")
print("=" * 60)

print("Dice metrics: PASS")
print("Model architecture: PASS")
print("Forward pass: PASS")
print("DiceCELoss: PASS")

print("\nPASS: Model and metrics integration verified.")

STEP 8.5B — MODEL AND METRICS SMOKE TEST

TEST 1A — Perfect prediction
Dice: {'WT': 1.0, 'TC': 1.0, 'ET': 1.0}
PASS: Perfect prediction gives Dice = 1.

TEST 1B — Both masks empty
Dice: {'WT': 1.0, 'TC': 1.0, 'ET': 1.0}
PASS: Empty-mask convention verified.

TEST 1C — False-positive ET
Dice: {'WT': 0.0, 'TC': 0.0, 'ET': 0.0}
PASS: False-positive ET gives ET Dice = 0.

TEST 2 — Model architecture
Parameters: 1,191,413
PASS: Model architecture verified.

TEST 3 — Forward pass
Input shape: (1, 4, 32, 32, 32)
Output shape: (1, 4, 32, 32, 32)
PASS: Forward pass verified.

TEST 4 — DiceCELoss
Loss: 2.349752
PASS: Loss computation verified.

STEP 8.5B SUMMARY
Dice metrics: PASS
Model architecture: PASS
Forward pass: PASS
DiceCELoss: PASS

PASS: Model and metrics integration verified.


In [ ]:
# STEP 8.5C - TRAINING ENGINE SMOKE TEST

import torch
from torch.utils.data import Dataset, DataLoader

print("=" * 60)
print("STEP 8.5C - TRAINING ENGINE SMOKE TEST")
print("=" * 60)

torch.manual_seed(42)

# ==========================================
# 1. CREATE SYNTHETIC DATASET
# ==========================================

class SyntheticPatchDataset(Dataset):
    def __len__(self):
        return 2

    def __getitem__(self, index):
        generator = torch.Generator().manual_seed(
            100 + index
        )

        # 4 MRI modalities, patch 32³
        image = torch.randn(
            4, 32, 32, 32,
            generator=generator,
            dtype=torch.float32,
        )

        label = torch.zeros(
            32, 32, 32,
            dtype=torch.long,
        )

        if index == 0:
            # Simulated tumor regions
            label[8:22, 8:22, 8:22] = 2
            label[11:19, 11:19, 11:19] = 1
            label[14:17, 14:17, 14:17] = 3

            center_type = "positive"

        else:
            center_type = "negative"

        return {
            "image": image,
            "label": label,
            "center_type": center_type,
        }


smoke_dataset = SyntheticPatchDataset()

smoke_loader = DataLoader(
    smoke_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
)

print("\nSynthetic dataset:")
print("Samples:", len(smoke_dataset))
print("Batches:", len(smoke_loader))
print("Patch size: 32 x 32 x 32")


# ==========================================
# 2. INITIALIZE A FRESH MODEL
# ==========================================

smoke_model = build_unet3d().to(device)

smoke_criterion = build_loss().to(device)

smoke_optimizer = torch.optim.AdamW(
    smoke_model.parameters(),
    lr=1e-4,
    weight_decay=1e-5,
)

smoke_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(device.type == "cuda"),
)

# Save initial weights for comparison
weights_before = (
    next(smoke_model.parameters())
    .detach()
    .clone()
)

print("\nTraining configuration:")
print("Device:", device)
print("Optimizer: AdamW")
print("Learning rate: 1e-4")
print("AMP enabled:", smoke_scaler.is_enabled())


# ==========================================
# 3. RUN ACTUAL TRAINING ENGINE
# ==========================================

smoke_stats = train_one_epoch(
    model=smoke_model,
    loader=smoke_loader,
    criterion=smoke_criterion,
    optimizer=smoke_optimizer,
    scaler=smoke_scaler,
    device=device,
    epoch=1,
    log_interval=1,
)


# ==========================================
# 4. VERIFY TRAINING RESULTS
# ==========================================

weights_after = (
    next(smoke_model.parameters())
    .detach()
    .clone()
)

weights_changed = not torch.equal(
    weights_before,
    weights_after,
)

assert torch.isfinite(
    torch.tensor(smoke_stats["mean_loss"])
).item()

assert (
    smoke_stats["optimizer_steps"]
    + smoke_stats["skipped_steps"]
    == len(smoke_loader)
)

assert smoke_stats["optimizer_steps"] >= 1

assert weights_changed, (
    "Model weights were not updated."
)

assert smoke_stats["positive_patches"] == 1
assert smoke_stats["tumor_patches"] == 1


# ==========================================
# 5. FINAL SUMMARY
# ==========================================

print("\n" + "=" * 60)
print("STEP 8.5C SUMMARY")
print("=" * 60)

print(
    "Mean training loss:",
    round(smoke_stats["mean_loss"], 6)
)

print(
    "Successful optimizer steps:",
    smoke_stats["optimizer_steps"]
)

print(
    "AMP skipped steps:",
    smoke_stats["skipped_steps"]
)

print(
    "Training time:",
    round(smoke_stats["epoch_time_seconds"], 2),
    "seconds"
)

print("Model weights updated:", weights_changed)
print("Positive patches:", smoke_stats["positive_patches"])
print("Tumor patches:", smoke_stats["tumor_patches"])

print("\nPASS: Training engine integration verified.")

STEP 8.5C — TRAINING ENGINE SMOKE TEST

Synthetic dataset:
Samples: 2
Batches: 2
Patch size: 32 x 32 x 32

Training configuration:
Device: cuda
Optimizer: AdamW
Learning rate: 1e-4
AMP enabled: True
Epoch 1 | Batch 1/2 | Mean loss: 2.6674 | AMP skipped: 0
Epoch 1 | Batch 2/2 | Mean loss: 2.7039 | AMP skipped: 0

AMP TRAINING STATISTICS
--------------------------------------------------
Successful optimizer steps: 2
Skipped optimizer steps: 0
Final AMP scale: 65536.0

STEP 8.5C SUMMARY
Mean training loss: 2.703932
Successful optimizer steps: 2
AMP skipped steps: 0
Training time: 1.11 seconds
Model weights updated: True
Positive patches: 1
Tumor patches: 1

PASS: Training engine integration verified.


In [ ]:
# STEP 8.5D - CHECKPOINT AND VALIDATION SMOKE TEST

from pathlib import Path

import torch
from torch.utils.data import Dataset

print("=" * 60)
print("STEP 8.5D - CHECKPOINT AND VALIDATION SMOKE TEST")
print("=" * 60)


# ==========================================
# TEST 1 - SAVE CHECKPOINT
# ==========================================

CHECKPOINT_PATH = Path(
    "/kaggle/working/integration_smoke_checkpoint.pt"
)

save_checkpoint(
    path=CHECKPOINT_PATH,
    model=smoke_model,
    optimizer=smoke_optimizer,
    scaler=smoke_scaler,
    epoch=1,
    global_step=2,
    best_val_dice=0.1234,
    config={
        "experiment": "integration_smoke_test",
        "patch_size": [32, 32, 32],
    },
)

assert CHECKPOINT_PATH.is_file()

print("\nTEST 1 - Save checkpoint")
print("Checkpoint:", CHECKPOINT_PATH.name)
print(
    "Size:",
    round(CHECKPOINT_PATH.stat().st_size / 1024**2, 2),
    "MiB",
)
print("PASS: Checkpoint saved.")


# ==========================================
# TEST 2 - LOAD INTO A NEW MODEL
# ==========================================

restored_model = build_unet3d().to(device)

restored_optimizer = torch.optim.AdamW(
    restored_model.parameters(),
    lr=1e-4,
    weight_decay=1e-5,
)

restored_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(device.type == "cuda"),
)

checkpoint = load_checkpoint(
    path=CHECKPOINT_PATH,
    model=restored_model,
    optimizer=restored_optimizer,
    scaler=restored_scaler,
    map_location="cpu",
)

assert checkpoint["epoch"] == 1
assert checkpoint["global_step"] == 2
assert checkpoint["best_val_dice"] == 0.1234

# Compare every saved model parameter and buffer
original_state = smoke_model.state_dict()
restored_state = restored_model.state_dict()

assert original_state.keys() == restored_state.keys()

for key in original_state:
    assert torch.equal(
        original_state[key].cpu(),
        restored_state[key].cpu(),
    ), f"Mismatch at: {key}"

# Optimizer state should also be restored
assert len(restored_optimizer.state) > 0

assert (
    restored_scaler.get_scale()
    == smoke_scaler.get_scale()
)

print("\nTEST 2 - Restore checkpoint")
print("Restored epoch:", checkpoint["epoch"])
print("Restored global_step:", checkpoint["global_step"])
print("Model state: identical")
print("Optimizer state: restored")
print("GradScaler scale: restored")
print("PASS: Checkpoint restored successfully.")


# ==========================================
# TEST 3 - SYNTHETIC FULL-VOLUME DATA
# ==========================================

class SyntheticVolumeDataset(Dataset):

    def __len__(self):
        return 1

    def __getitem__(self, index):
        return {
            "image": torch.zeros(
                4, 16, 16, 16,
                dtype=torch.float32,
            ),
            "label": torch.zeros(
                16, 16, 16,
                dtype=torch.long,
            ),
            "subject_id": "synthetic_001",
        }


class BackgroundOnlyModel(torch.nn.Module):

    def forward(self, x):
        logits = torch.zeros(
            x.shape[0],
            4,
            *x.shape[2:],
            device=x.device,
            dtype=x.dtype,
        )

        # Force background (class 0) everywhere.
        logits[:, 0] = 10.0

        return logits


synthetic_val_dataset = SyntheticVolumeDataset()

background_model = BackgroundOnlyModel().to(device)

print("\nTEST 3 - Synthetic validation data")
print("Subjects:", len(synthetic_val_dataset))
print("Image shape: (4, 16, 16, 16)")
print("Ground truth: all background")
print("Prediction: all background")


# ==========================================
# TEST 4 - SLIDING-WINDOW VALIDATION
# ==========================================

validation_summary, validation_results = validate_model(
    model=background_model,
    dataset=synthetic_val_dataset,
    device=device,
    roi_size=(8, 8, 8),
    overlap=0.25,
    max_subjects=1,
)

assert len(validation_results) == 1
assert validation_summary["num_subjects"] == 1

for region in ("WT", "TC", "ET"):
    assert abs(
        validation_summary[region] - 1.0
    ) < 1e-8

assert validation_summary["mean_dice"] == 1.0
assert validation_summary["num_empty_ET_targets"] == 1

print("\nTEST 4 - Full-volume validation")
print("WT Dice:", validation_summary["WT"])
print("TC Dice:", validation_summary["TC"])
print("ET Dice:", validation_summary["ET"])
print("Mean Dice:", validation_summary["mean_dice"])
print("PASS: Sliding-window validation verified.")


# ==========================================
# CLEAN UP TEMPORARY CHECKPOINT
# ==========================================

CHECKPOINT_PATH.unlink()

print("\nTemporary checkpoint removed.")


# ==========================================
# FINAL SUMMARY
# ==========================================

print("\n" + "=" * 60)
print("STEP 8.5D SUMMARY")
print("=" * 60)

print("Checkpoint save: PASS")
print("Checkpoint restore: PASS")
print("Model weights identical: PASS")
print("Optimizer / GradScaler restore: PASS")
print("Sliding-window inference: PASS")
print("WT / TC / ET Dice: PASS")

print("\nPASS: Checkpoint and validation integration verified.")

STEP 8.5D — CHECKPOINT AND VALIDATION SMOKE TEST

TEST 1 — Save checkpoint
Checkpoint: integration_smoke_checkpoint.pt
Size: 13.71 MiB
PASS: Checkpoint saved.

TEST 2 — Restore checkpoint
Restored epoch: 1
Restored global_step: 2
Model state: identical
Optimizer state: restored
GradScaler scale: restored
PASS: Checkpoint restored successfully.

TEST 3 — Synthetic validation data
Subjects: 1
Image shape: (4, 16, 16, 16)
Ground truth: all background
Prediction: all background
Validated 1/1 subjects

TEST 4 — Full-volume validation
WT Dice: 1.0
TC Dice: 1.0
ET Dice: 1.0
Mean Dice: 1.0
PASS: Sliding-window validation verified.

Temporary checkpoint removed.

STEP 8.5D SUMMARY
Checkpoint save: PASS
Checkpoint restore: PASS
Model weights identical: PASS
Optimizer / GradScaler restore: PASS
Sliding-window inference: PASS
WT / TC / ET Dice: PASS

PASS: Checkpoint and validation integration verified.
